# 03 · Power BI dashboard (Direct Lake) — built as code
Attach **lh_fraud** as default lakehouse, then **Run all** (≈ 3–5 min). Re-runnable: it overwrites the BI tables, the model and the report.

1. `pbi_*` Delta tables shaped for reporting (from the Gold tables of notebook 02)
2. Direct Lake semantic model **sm_fraud_copilot** + DAX measures + relationship
3. Report **Fraud Copilot — Dashboard** (4 pages, custom theme)

No Power BI Copilot is used (capacity is judged).

In [ ]:
%%configure -f
{"defaultLakehouse": {"name": "lh_fraud"}}

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import json
import pandas as pd
from pyspark.sql import functions as F

def save(df, name):
    (df if not isinstance(df, pd.DataFrame) else spark.createDataFrame(df)) \
        .write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)

q = spark.table("gold_alert_queue")
print("queue columns:", q.columns)
print("risk levels:", [r[0] for r in q.select("risk_level").distinct().collect()])

SCEN = {"CARD_TESTING": "Test de carte", "ACCOUNT_TAKEOVER": "Prise de contrôle de compte",
        "CARTE_CLONEE_ETRANGER": "Carte clonée à l'étranger", "SKIMMING_DAB": "Skimming DAB",
        "CARTE_VOLEE_SANS_CONTACT": "Carte volée sans contact", "FRAUDE_EN_LIGNE_DISCRETE": "Fraude en ligne discrète"}
scen_map = F.create_map([F.lit(x) for kv in SCEN.items() for x in kv])

alerts = (q.withColumn("alert_date", F.to_date("timestamp"))
           .withColumn("alert_hour", F.hour("timestamp"))
           .withColumn("alert_month", F.date_format("timestamp", "yyyy-MM"))
           .withColumn("periode", F.when(F.col("period") == "test", "Flux live (août–sept.)").otherwise("Validation (juin–juil.)"))
           .withColumn("verite", F.when(F.col("period") == "test", "Inconnue (flux live)")
                                  .when(F.col("is_fraud") == 1, "Fraude confirmée").otherwise("Légitime"))
           .withColumn("vu_par_regles", F.when(F.col("legacy_alert") == 1, "Aussi alerté par les règles")
                                         .otherwise("Manqué par les règles"))
           .withColumn("typologie", F.coalesce(scen_map[F.col("fraud_scenario")], F.col("fraud_scenario"), F.lit("—")))
           .withColumn("bande_score", F.when(F.col("score_model") >= 0.9, "1 · ≥ 0,90").when(F.col("score_model") >= 0.7, "2 · 0,70–0,90")
                                       .when(F.col("score_model") >= 0.5, "3 · 0,50–0,70").otherwise("4 · < 0,50"))
           .withColumn("is_fraud_n", F.coalesce(F.col("is_fraud").cast("int"), F.lit(0)))
           .withColumn("amount_tnd", F.col("amount_tnd").cast("double")))
save(alerts, "pbi_alerts")

ex = spark.table("gold_alert_explanations")
factors = (ex.withColumn("factor_kind", F.when(F.col("source").startswith("rules"), "Règle métier")
                                        .when(F.lower(F.col("source")).contains("shap"), "Modèle (SHAP)")
                                        .otherwise(F.col("source")))
             .select("transaction_id", "factor", "factor_kind", "points", "observed", "usual"))
save(factors, "pbi_factors")
print("factor kinds:", [r[0] for r in factors.select("factor_kind").distinct().collect()])

In [ ]:
m = {r["metric"]: float(r["value"]) for r in spark.table("gold_model_metrics").collect()}
g = lambda k, d=0.0: m.get(k, d)
vr, vf = g("valid_rows"), g("valid_frauds")
# exact top-1 % counts from the validation alerts themselves (no rounding drift vs the app / STATUS)
_va = q.filter(F.col("period") == "valid").agg(F.count("*").alias("n"), F.sum("is_fraud").alias("f")).first()
top1_alerts, top1_frauds = int(_va["n"]), int(_va["f"] or 0)
top05_alerts, top05_frauds = round(vr * 0.005), round(g("model_top0.5pct_recall") * vf)

compare = pd.DataFrame([
    ("1 · Moteur de règles actuel", g("legacy_alerts"), g("frauds_caught_legacy"), g("legacy_recall"), g("legacy_precision")),
    ("2 · Modèle, même volume d'alertes", g("legacy_alerts"), g("frauds_caught_model_same_budget"),
     g("model_same_budget_as_legacy_recall"), g("model_same_budget_as_legacy_precision")),
    ("3 · Modèle, top 1 % (capacité analystes)", top1_alerts, top1_frauds, g("model_top1pct_recall"), g("model_top1pct_precision")),
    ("4 · Modèle, top 0,5 %", top05_alerts, top05_frauds, g("model_top0.5pct_recall"), g("model_top0.5pct_precision")),
], columns=["configuration", "alerts", "frauds", "recall", "precision"])
compare["false_alerts"] = compare.alerts - compare.frauds
save(compare, "pbi_compare")

kpi = pd.DataFrame([{
    "frauds_model": g("frauds_caught_model_same_budget"), "frauds_rules": g("frauds_caught_legacy"),
    "recall_model": g("model_same_budget_as_legacy_recall"), "recall_rules": g("legacy_recall"),
    "precision_top1": g("model_top1pct_precision"), "precision_rules": g("legacy_precision"),
    "false_alerts_avoided": (g("legacy_alerts") - g("frauds_caught_legacy")) - (top1_alerts - top1_frauds),
    "extra_frauds": top1_frauds - g("frauds_caught_legacy"),
    "pr_auc_model": g("pr_auc_model"), "pr_auc_rules": g("pr_auc_rules"),
    "valid_frauds": vf, "valid_rows": vr}])
save(kpi, "pbi_kpi")
print(compare.to_string()); print(kpi.T)

rt = spark.table("gold_model_recall_by_type").toPandas()
rt["typologie"] = rt.fraud_scenario.map(SCEN).fillna(rt.fraud_scenario)
save(rt, "pbi_recall_wide")
long = pd.concat([rt.assign(moteur="Modèle", recall=rt.recall_model, caught=rt.caught_top1pct),
                  rt.assign(moteur="Règles actuelles", recall=rt.recall_legacy, caught=rt.caught_legacy)])
save(long[["typologie", "moteur", "recall", "caught", "frauds"]], "pbi_recall_type")

save(spark.table("gold_model_leaderboard"), "pbi_leaderboard")
LABEL = {"f_speed_kmh": "Vitesse de déplacement", "f_amount": "Montant", "merchant_category": "Catégorie marchande",
         "channel": "Canal", "f_secs_since_prev": "Temps depuis l'opération précédente", "f_new_merchant": "Nouveau commerçant",
         "transaction_type": "Type d'opération", "entry_mode": "Mode de saisie", "f_n_before": "Ancienneté (nb opérations)",
         "f_key_contactless_24h": "Sans contact sur 24 h", "f_is_foreign": "Opération à l'étranger",
         "merchant_type": "Type de commerçant", "f_same_country_7d": "Opérations dans ce pays (7 j)",
         "f_dist_prev_km": "Distance depuis l'opération précédente", "f_amount_z": "Écart au montant habituel (z)",
         "f_amount_over_limit": "Montant / plafond", "f_amount_ratio": "Montant / moyenne habituelle",
         "f_merchant_frauds_30d": "Commerçant lié à des fraudes (30 j)", "f_night_share_before": "Habitude d'opérer la nuit",
         "f_key_new_merchants_24h": "Nouveaux commerçants sur 24 h"}
imp = spark.table("gold_model_importance").toPandas()
imp["variable"] = imp.feature.map(LABEL).fillna(imp.feature)
save(imp, "pbi_importance")
print("BI tables written")

In [ ]:
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs import directlake

DS, LH = "sm_fraud_copilot", "lh_fraud"
TABLES = ["pbi_alerts", "pbi_factors", "pbi_kpi", "pbi_compare", "pbi_recall_type", "pbi_recall_wide",
          "pbi_leaderboard", "pbi_importance"]
REBUILD_MODEL = False   # True only if the pbi_* table list/columns change (keeps the report bound otherwise)
if REBUILD_MODEL or DS not in set(fabric.list_datasets()["Dataset Name"]):
    # lh_fraud is schema-enabled: tables live under dbo
    directlake.generate_direct_lake_semantic_model(dataset=DS, tables=["dbo." + t for t in TABLES], source=LH,
                                                   source_type="Lakehouse", overwrite=True, refresh=False)
print("model ready:", DS)

In [ ]:
from sempy_labs.tom import connect_semantic_model

MEASURES = {
    "pbi_alerts": [
        ("Alertes", "COUNTROWS(pbi_alerts)", "#,0"),
        ("Montant en jeu (TND)", "SUM(pbi_alerts[amount_tnd])", "#,0"),
        ("Score moyen", "AVERAGE(pbi_alerts[score_model])", "0.000"),
        ("Alertes risque élevé", "CALCULATE([Alertes], pbi_alerts[risk_level] = \"ELEVE\")", "#,0"),
        ("Manquées par les règles", "CALCULATE([Alertes], pbi_alerts[legacy_alert] = 0)", "#,0"),
        ("Fraudes confirmées", "SUM(pbi_alerts[is_fraud_n])", "#,0"),
        ("Précision des alertes (validation)",
         "DIVIDE([Fraudes confirmées], CALCULATE([Alertes], pbi_alerts[period] = \"valid\"))", "0.0%"),
    ],
    "pbi_factors": [("Occurrences", "COUNTROWS(pbi_factors)", "#,0")],
    "pbi_kpi": [
        ("Fraudes détectées · modèle", "SUM(pbi_kpi[frauds_model])", "#,0"),
        ("Fraudes détectées · règles", "SUM(pbi_kpi[frauds_rules])", "#,0"),
        ("Rappel · modèle", "SUM(pbi_kpi[recall_model])", "0.0%"),
        ("Rappel · règles", "SUM(pbi_kpi[recall_rules])", "0.0%"),
        ("Précision top 1 %", "SUM(pbi_kpi[precision_top1])", "0.0%"),
        ("Précision · règles", "SUM(pbi_kpi[precision_rules])", "0.0%"),
        ("Fausses alertes évitées", "SUM(pbi_kpi[false_alerts_avoided])", "#,0"),
        ("Fraudes en plus", "SUM(pbi_kpi[extra_frauds])", "#,0"),
        ("PR-AUC · modèle", "SUM(pbi_kpi[pr_auc_model])", "0.000"),
        ("Gain de rappel (×)", "DIVIDE([Rappel · modèle], [Rappel · règles])", "0.0\"×\""),
    ],
    "pbi_compare": [
        ("Alertes générées", "SUM(pbi_compare[alerts])", "#,0"),
        ("Fraudes détectées", "SUM(pbi_compare[frauds])", "#,0"),
        ("Fausses alertes", "SUM(pbi_compare[false_alerts])", "#,0"),
    ],
    "pbi_recall_type": [("Rappel", "AVERAGE(pbi_recall_type[recall])", "0%")],
    "pbi_leaderboard": [("PR-AUC", "SUM(pbi_leaderboard[pr_auc])", "0.000"),
                        ("Rappel top 1 % (modèles)", "SUM(pbi_leaderboard[top1pct_recall])", "0.0%")],
    "pbi_importance": [("Importance", "SUM(pbi_importance[gain_share])", "0.0%")],
    "pbi_recall_wide": [("Fraudes attrapées (modèle)", "SUM(pbi_recall_wide[caught_top1pct])", "#,0")],
}
PCT_COLS = {"pbi_compare": ["recall", "precision"], "pbi_recall_wide": ["recall_model", "recall_legacy"]}
INT_COLS = {"pbi_compare": ["alerts", "frauds", "false_alerts"],
            "pbi_recall_wide": ["frauds", "caught_top1pct", "caught_legacy"], "pbi_alerts": ["amount_tnd"]}
NUM_FMT = {("pbi_alerts", "score_model"): "0.000"}

with connect_semantic_model(dataset=DS, readonly=False) as tom:
    existing = {(m.Parent.Name, m.Name) for m in tom.all_measures()}
    for t, ms in MEASURES.items():
        for name, expr, fmt in ms:
            if (t, name) in existing:
                tom.model.Tables[t].Measures[name].Expression = expr
                tom.model.Tables[t].Measures[name].FormatString = fmt
            else:
                tom.add_measure(table_name=t, measure_name=name, expression=expr, format_string=fmt)
    for t, cols in PCT_COLS.items():
        for c in cols:
            tom.model.Tables[t].Columns[c].FormatString = "0.0%"
    for t, cols in INT_COLS.items():
        for c in cols:
            tom.model.Tables[t].Columns[c].FormatString = "#,0"
    for (t, c), f in NUM_FMT.items():
        tom.model.Tables[t].Columns[c].FormatString = f
    if not any(r.FromTable.Name == "pbi_factors" for r in tom.model.Relationships):
        tom.add_relationship(from_table="pbi_factors", from_column="transaction_id", to_table="pbi_alerts",
                             to_column="transaction_id", from_cardinality="Many", to_cardinality="One")
labs.refresh_semantic_model(dataset=DS)
print("measures + relationship OK")

In [ ]:
import json, uuid

# ---- palette (dark navy, inspired by the reference design)
BG, CARD, BAR, LINE = "#0B1426", "#14213A", "#18284A", "#22324F"
WHITE, LIGHT, MUTED = "#FFFFFF", "#D9E2F2", "#8A9BB8"
BLUE, BLUE2, SLATE, ORANGE, RED, GREEN = "#4C7CF3", "#8FB0FF", "#5B6B8C", "#F59E0B", "#F43F5E", "#22C55E"

def lit(v):
    return {"expr": {"Literal": {"Value": v}}}

def s(text):  # Power BI string literal
    return lit("'" + str(text).replace("'", "''") + "'")

def color(hex_):
    return {"solid": {"color": s(hex_)}}

def num(v):
    return lit("%sD" % v)

T, F_ = lit("true"), lit("false")
C = lambda t, c: ("Column", t, c)
M = lambda t, m: ("Measure", t, m)

def by_value(table, column, mapping):
    """dataPoint colours for each category value (series / donut slices)."""
    return [{"properties": {"fill": color(c)},
             "selector": {"data": [{"scopeId": {"Comparison": {"ComparisonKind": 0,
                 "Left": {"Column": {"Expression": {"SourceRef": {"Entity": table}}, "Property": column}},
                 "Right": {"Literal": {"Value": "'" + v.replace("'", "''") + "'"}}}}}]}}
            for v, c in mapping.items()]

def by_measure(table, mapping):
    return [{"properties": {"fill": color(c)}, "selector": {"metadata": "%s.%s" % (table, m)}} for m, c in mapping.items()]

def chart_style(fill=BLUE, legend=False, labels=True, extra_points=None):
    o = {"categoryAxis": [{"properties": {"labelColor": color(MUTED), "fontSize": num(9), "showAxisTitle": F_,
                                          "gridlineShow": F_}}],
         "valueAxis": [{"properties": {"labelColor": color(MUTED), "fontSize": num(9), "showAxisTitle": F_,
                                       "gridlineShow": T, "gridlineColor": color(LINE)}}],
         "legend": [{"properties": {"show": T if legend else F_, "position": s("Top"), "labelColor": color(LIGHT), "showTitle": F_,
                                    "fontSize": num(9)}}],
         "labels": [{"properties": {"show": T if labels else F_, "color": color(LIGHT), "fontSize": num(9), "labelDisplayUnits": num(1)}}],
         "dataPoint": [{"properties": {"fill": color(fill)}}] + (extra_points or [])}
    return o

def _container(x, y, w, h, z, single):
    cfg = {"name": uuid.uuid4().hex[:20],
           "layouts": [{"id": 0, "position": {"x": x, "y": y, "z": z, "width": w, "height": h}}],
           "singleVisual": single}
    return {"x": x, "y": y, "z": z, "width": w, "height": h, "config": json.dumps(cfg), "filters": "[]"}

def panel(title=None, subtitle=None, bg=CARD):
    vc = {"background": [{"properties": {"show": T, "color": color(bg), "transparency": num(0)}}],
          "border": [{"properties": {"show": T, "color": color(bg), "radius": num(14)}}],
          "dropShadow": [{"properties": {"show": F_}}],
          "visualHeader": [{"properties": {"show": F_}}]}
    if title:
        vc["title"] = [{"properties": {"show": T, "text": s(title), "fontColor": color(WHITE), "fontSize": num(12),
                                       "bold": T, "fontFamily": s("Segoe UI Semibold")}}]
    else:
        vc["title"] = [{"properties": {"show": F_}}]
    if subtitle:
        vc["subTitle"] = [{"properties": {"show": T, "text": s(subtitle), "fontColor": color(MUTED), "fontSize": num(9)}}]
    return vc

PAGE_NAMES = ["Vue exécutive", "File d'alertes", "Pourquoi ?", "Typologies"]

class Page:
    def __init__(self, name, idx, title, subtitle):
        self.name, self.idx, self.title, self.items, self.z = name, idx, PAGE_NAMES[idx], [], 0
        # brand bar (left) + page navigation text (right), like the reference header
        self._text(16, 12, 1248, 44, [[("◆  FRAUD COPILOT", {"fontSize": "12pt", "fontWeight": "bold", "color": WHITE}),
                                       ("    ·  Banque de détail · détection de fraude", {"fontSize": "9pt", "color": MUTED})]],
                   bg=BAR)
        nav = []
        for i, p in enumerate(PAGE_NAMES):
            st = {"fontSize": "10pt", "color": BLUE2 if i == idx else MUTED}
            if i == idx:
                st["fontWeight"] = "bold"
            nav.append((p + ("" if i == len(PAGE_NAMES) - 1 else "      "), st))
        self._text(640, 16, 610, 36, [nav], align="right", bg=BAR)
        self._text(16, 62, 1248, 44, [[(title, {"fontSize": "13pt", "fontWeight": "bold", "color": WHITE}),
                                       ("     " + subtitle, {"fontSize": "9pt", "color": MUTED})]], bg=BG)

    def _z(self):
        self.z += 1000
        return self.z

    def _text(self, x, y, w, h, paragraphs, align=None, bg=CARD):
        paras = []
        for runs in paragraphs:
            p = {"textRuns": [{"value": v, "textStyle": st} for v, st in runs]}
            if align:
                p["horizontalTextAlignment"] = align
            paras.append(p)
        vc = panel(bg=bg)
        if bg == BAR:
            vc["border"] = [{"properties": {"show": T, "color": color(BAR), "radius": num(10)}}]
        single = {"visualType": "textbox", "drillFilterOtherVisuals": True,
                  "objects": {"general": [{"properties": {"paragraphs": paras}}]}, "vcObjects": vc}
        self.items.append(_container(x, y, w, h, self._z(), single))

    def visual(self, vtype, x, y, w, h, roles, title=None, subtitle=None, objects=None, order_by=None, desc=True,
               names=None):
        froms, select, proj = {}, [], {}
        for role, fields in roles.items():
            proj[role] = []
            for kind, t, f in fields:
                a = froms.setdefault(t, "t%d" % len(froms))
                name = "%s.%s" % (t, f)
                if all(x_["Name"] != name for x_ in select):
                    select.append({kind: {"Expression": {"SourceRef": {"Source": a}}, "Property": f},
                                   "Name": name, "NativeReferenceName": f})
                proj[role].append({"queryRef": name})
        query = {"Version": 2, "From": [{"Name": a, "Entity": t, "Type": 0} for t, a in froms.items()],
                 "Select": select}
        if order_by:
            kind, t, f = order_by
            query["OrderBy"] = [{"Direction": 2 if desc else 1,
                                 "Expression": {kind: {"Expression": {"SourceRef": {"Source": froms[t]}}, "Property": f}}}]
        single = {"visualType": vtype, "projections": proj, "prototypeQuery": query,
                  "drillFilterOtherVisuals": True, "objects": objects or {}, "vcObjects": panel(title, subtitle)}
        if names:
            single["columnProperties"] = {"%s.%s" % (t, f): {"displayName": dn} for (t, f), dn in names.items()}
        self.items.append(_container(x, y, w, h, self._z(), single))

    def card(self, x, y, w, h, measure, label, value_color=WHITE):
        objects = {"labels": [{"properties": {"color": color(value_color), "fontSize": num(26), "labelDisplayUnits": num(1),
                                              "fontFamily": s("Segoe UI Semibold")}}],
                   "categoryLabels": [{"properties": {"show": T, "color": color(MUTED), "fontSize": num(10)}}]}
        t, m = measure[1], measure[2]
        self.visual("card", x, y, w, h, {"Values": [measure]}, objects=objects, names={(t, m): label})

    def slicer(self, x, y, w, h, column, title):
        objects = {"data": [{"properties": {"mode": s("Dropdown")}}],
                   "header": [{"properties": {"show": T, "text": s(title), "fontColor": color(MUTED), "fontSize": num(9)}}],
                   "items": [{"properties": {"fontColor": color(LIGHT), "background": color(CARD), "fontSize": num(10)}}],
                   "general": [{"properties": {"outlineColor": color(LINE)}}]}
        self.visual("slicer", x, y, w, h, {"Values": [column]}, objects=objects)

    def table(self, x, y, w, h, columns, title, subtitle=None, names=None, order_by=None):
        objects = {"grid": [{"properties": {"gridVertical": F_, "gridHorizontal": T, "gridHorizontalColor": color(LINE),
                                            "outlineColor": color(LINE), "rowPadding": num(4), "textSize": num(10)}}],
                   "columnHeaders": [{"properties": {"fontColor": color(MUTED), "backColor": color(CARD),
                                                     "outlineStyle": num(0), "fontSize": num(9)}}],
                   "values": [{"properties": {"fontColorPrimary": color(LIGHT), "backColorPrimary": color(CARD),
                                              "fontColorSecondary": color(LIGHT), "backColorSecondary": color(CARD),
                                              "outlineStyle": num(0)}}],
                   "total": [{"properties": {"totals": F_}}]}
        self.visual("tableEx", x, y, w, h, {"Values": columns}, title, subtitle, objects, order_by=order_by, names=names)

    def section(self, i):
        page_bg = {"background": [{"properties": {"color": color(BG), "transparency": num(0)}}],
                   "outspace": [{"properties": {"color": color(BG), "transparency": num(0)}}]}
        return {"name": self.name, "displayName": self.title, "filters": "[]", "ordinal": i,
                "visualContainers": self.items, "config": json.dumps({"objects": page_bg}),
                "displayOption": 1, "width": 1280, "height": 720}

A, K, P, R, L, I, X, W = ("pbi_alerts", "pbi_kpi", "pbi_compare", "pbi_recall_type", "pbi_leaderboard",
                          "pbi_importance", "pbi_factors", "pbi_recall_wide")
VERITE = {"Fraude confirmée": RED, "Légitime": SLATE, "Inconnue (flux live)": BLUE}
CHANNEL = {"TPE": BLUE, "DAB": BLUE2, "EN_LIGNE": ORANGE, "WEB": GREEN, "MOBILE": "#A78BFA"}

# ---------------- Page 1: executive view
p1 = Page("p1exec", 0, "Vue exécutive · le copilote face au moteur de règles actuel",
          "Validation hors échantillon juin–juillet 2026 · même volume d'alertes · le modèle score, l'IA explique, l'analyste décide")
for i, (mm, lab, col) in enumerate([
        (M(K, "Fraudes détectées · modèle"), "Fraudes détectées · modèle", WHITE),
        (M(K, "Fraudes détectées · règles"), "Fraudes détectées · règles actuelles", MUTED),
        (M(K, "Fausses alertes évitées"), "Fausses alertes évitées", WHITE),
        (M(K, "Précision top 1 %"), "Précision des alertes (top 1 %)", WHITE),
        (M(K, "Gain de rappel (×)"), "Fraudes trouvées en plus (rappel)", BLUE2)]):
    p1.card(16 + i * 252, 112, 240, 96, mm, lab, col)
p1.visual("clusteredBarChart", 16, 220, 616, 240,
          {"Category": [C(R, "typologie")], "Series": [C(R, "moteur")], "Y": [M(R, "Rappel")]},
          "Rappel par typologie", "Part des fraudes détectées : modèle vs règles actuelles",
          chart_style(legend=True, extra_points=by_value(R, "moteur", {"Modèle": BLUE, "Règles actuelles": SLATE})))
p1.visual("clusteredColumnChart", 644, 220, 620, 240,
          {"Category": [C(P, "configuration")], "Y": [M(P, "Fraudes détectées"), M(P, "Fausses alertes")]},
          "Fraudes vs fausses alertes", "Par configuration d'alerte (validation)",
          chart_style(legend=True, extra_points=by_measure(P, {"Fraudes détectées": BLUE, "Fausses alertes": ORANGE})))
p1.table(16, 472, 760, 236,
         [C(P, "configuration"), C(P, "alerts"), C(P, "frauds"), C(P, "false_alerts"), C(P, "recall"), C(P, "precision")],
         "Tableau de comparaison", "Validation juin–juillet 2026, hors échantillon",
         names={(P, "configuration"): "Configuration", (P, "alerts"): "Alertes", (P, "frauds"): "Fraudes",
                (P, "false_alerts"): "Fausses alertes", (P, "recall"): "Rappel", (P, "precision"): "Précision"})
p1.visual("clusteredBarChart", 788, 472, 476, 236, {"Category": [C(L, "model")], "Y": [M(L, "PR-AUC")]},
          "PR-AUC par modèle testé", "Champion : LightGBM (score + explication SHAP)", chart_style(),
          order_by=M(L, "PR-AUC"))

# ---------------- Page 2: alert queue
p2 = Page("p2queue", 1, "File d'alertes · priorisation pour les analystes",
          "Top 1 % des scores par période · flux live = août–septembre 2026 (sans label)")
for i, (col_, t_) in enumerate([("periode", "Période"), ("risk_level", "Niveau de risque"), ("channel", "Canal"),
                                ("vu_par_regles", "Vu par les règles ?"), ("verite", "Vérité terrain")]):
    p2.slicer(16 + i * 252, 110, 240, 64, C(A, col_), t_)
for i, (mm, lab, col) in enumerate([(M(A, "Alertes"), "Alertes", WHITE), (M(A, "Alertes risque élevé"), "Risque élevé", RED),
                                    (M(A, "Manquées par les règles"), "Manquées par les règles", BLUE2),
                                    (M(A, "Montant en jeu (TND)"), "Montant en jeu (TND)", WHITE),
                                    (M(A, "Précision des alertes (validation)"), "Précision (validation)", GREEN)]):
    p2.card(16 + i * 252, 180, 240, 88, mm, lab, col)
p2.visual("areaChart", 16, 280, 760, 204, {"Category": [C(A, "alert_date")], "Y": [M(A, "Alertes")]},
          "Alertes par jour", "Volume quotidien de la file", chart_style(labels=False))
p2.visual("donutChart", 788, 280, 476, 204, {"Category": [C(A, "channel")], "Y": [M(A, "Alertes")]},
          "Alertes par canal", "TPE = magasin · DAB = distributeur",
          {"legend": [{"properties": {"show": T, "position": s("Right"), "labelColor": color(LIGHT), "showTitle": F_}}],
           "labels": [{"properties": {"show": T, "color": color(LIGHT), "labelDisplayUnits": num(1),
                                      "labelStyle": s("Percent of total")}}],
           "dataPoint": by_value(A, "channel", CHANNEL)})
p2.table(16, 496, 760, 212,
         [C(A, "transaction_id"), C(A, "alert_date"), C(A, "customer_id"), C(A, "amount_tnd"), C(A, "channel"),
          C(A, "country"), C(A, "score_model"), C(A, "risk_level"), C(A, "top_factors")],
         "Alertes à traiter en priorité", "Triées par score du modèle",
         names={(A, "transaction_id"): "Transaction", (A, "alert_date"): "Date", (A, "customer_id"): "Client",
                (A, "amount_tnd"): "Montant (TND)", (A, "channel"): "Canal", (A, "country"): "Pays",
                (A, "score_model"): "Score", (A, "risk_level"): "Risque", (A, "top_factors"): "Facteurs principaux"},
         order_by=C(A, "score_model"))
p2.visual("clusteredBarChart", 788, 496, 476, 212, {"Category": [C(A, "merchant_category")], "Y": [M(A, "Alertes")]},
          "Alertes par catégorie marchande", None, chart_style(), order_by=M(A, "Alertes"))

# ---------------- Page 3: explainability
p3 = Page("p3why", 2, "Pourquoi ? · explicabilité des alertes",
          "Facteurs stockés par alerte (observé vs habituel) : c'est ce que lit le copilote pour expliquer, sans rien inventer")
p3.slicer(16, 110, 300, 64, C(A, "periode"), "Période")
p3.slicer(328, 110, 300, 64, C(A, "risk_level"), "Niveau de risque")
p3.slicer(640, 110, 300, 64, C(X, "factor_kind"), "Source du facteur")
p3.visual("clusteredBarChart", 16, 180, 616, 280, {"Category": [C(X, "factor")], "Y": [M(X, "Occurrences")]},
          "Facteurs de risque les plus fréquents", "Nombre d'alertes où le facteur apparaît", chart_style(),
          order_by=M(X, "Occurrences"))
p3.visual("clusteredBarChart", 644, 180, 620, 280, {"Category": [C(I, "variable")], "Y": [M(I, "Importance")]},
          "Variables les plus utilisées par le modèle", "Part du gain LightGBM", chart_style(fill=BLUE2),
          order_by=M(I, "Importance"))
p3.visual("clusteredColumnChart", 16, 472, 616, 236, {"Category": [C(A, "alert_hour")], "Y": [M(A, "Alertes")]},
          "Alertes par heure", "La nuit est sur-représentée dans les fraudes", chart_style(labels=False))
p3.visual("columnChart", 644, 472, 620, 236,
          {"Category": [C(A, "bande_score")], "Series": [C(A, "verite")], "Y": [M(A, "Alertes")]},
          "Score vs vérité terrain", "Plus le score est haut, plus la fraude est confirmée",
          chart_style(legend=True, extra_points=by_value(A, "verite", VERITE)))

# ---------------- Page 4: fraud types & validation
p4 = Page("p4types", 3, "Typologies · ce que le modèle attrape, et ce qu'il rate",
          "Validation juin–juillet 2026 · rappel par typologie, y compris là où les règles restent utiles")
p4.table(16, 112, 760, 250,
         [C(W, "typologie"), C(W, "frauds"), C(W, "caught_top1pct"), C(W, "caught_legacy"), C(W, "recall_model"),
          C(W, "recall_legacy")],
         "Fraudes par typologie", "Détectées par le modèle (top 1 %) vs par les règles actuelles",
         names={(W, "typologie"): "Typologie", (W, "frauds"): "Fraudes", (W, "caught_top1pct"): "Modèle",
                (W, "caught_legacy"): "Règles", (W, "recall_model"): "Rappel modèle", (W, "recall_legacy"): "Rappel règles"})
p4.visual("donutChart", 788, 112, 476, 250, {"Category": [C(A, "vu_par_regles")], "Y": [M(A, "Fraudes confirmées")]},
          "Fraudes nouvelles vs déjà vues", "Fraudes confirmées dans nos alertes",
          {"legend": [{"properties": {"show": T, "position": s("Bottom"), "labelColor": color(LIGHT), "showTitle": F_}}],
           "labels": [{"properties": {"show": T, "color": color(LIGHT), "labelDisplayUnits": num(1)}}],
           "dataPoint": by_value(A, "vu_par_regles", {"Manqué par les règles": BLUE, "Aussi alerté par les règles": SLATE})})
p4.visual("clusteredBarChart", 16, 374, 616, 334, {"Category": [C(W, "typologie")], "Y": [M(W, "Fraudes attrapées (modèle)")]},
          "Fraudes attrapées par typologie", "Top 1 % des scores, validation juin–juillet", chart_style(),
          order_by=M(W, "Fraudes attrapées (modèle)"))
p4.visual("columnChart", 644, 374, 620, 334,
          {"Category": [C(A, "alert_month")], "Series": [C(A, "verite")], "Y": [M(A, "Alertes")]},
          "Alertes par mois", "Juin–juillet : validation · août–septembre : flux live",
          chart_style(legend=True, extra_points=by_value(A, "verite", VERITE)))

PAGES = [p1, p2, p3, p4]
REPORT_JSON = {
    "config": json.dumps({"version": "5.43", "themeCollection": {"baseTheme": {"name": "CY24SU06", "version": "5.55", "type": 2}},
                          "activeSectionIndex": 0, "defaultDrillFilterOtherVisuals": True,
                          "settings": {"useStylableVisualContainerHeader": True}}),
    "layoutOptimization": 0, "resourcePackages": [],
    "sections": [p.section(i) for i, p in enumerate(PAGES)],
}
THEME = {"name": "Fraud Copilot Dark", "dataColors": [BLUE, BLUE2, ORANGE, GREEN, RED, "#A78BFA", SLATE],
         "background": CARD, "foreground": WHITE, "tableAccent": BLUE}
print("pages:", [p.title for p in PAGES], "| visuals:", sum(len(p.items) for p in PAGES))

In [ ]:
import inspect
from sempy_labs import report as rep

def call(fn, **kw):  # pass only the arguments this library version accepts
    params = inspect.signature(fn).parameters
    print(fn.__name__, inspect.signature(fn))
    return fn(**{k: v for k, v in kw.items() if k in params})

RPT = "Fraud Copilot — Dashboard"
reports = fabric.list_reports()
if RPT in set(reports.get("Name", [])):
    call(rep.update_report_from_reportjson, report=RPT, report_json=REPORT_JSON)
    print("report updated")
else:
    call(rep.create_report_from_reportjson, report=RPT, dataset=DS, report_json=REPORT_JSON, theme=THEME)
    print("report created")
print(fabric.list_reports()[["Name"]].to_string())

In [ ]:
mssparkutils.session.stop()   # free the capacity